In [20]:
import os
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder
from groq import Groq

# Load environment variables
load_dotenv()
pinecone_key = os.environ.get("PINECONE_API_KEY")
groq_key = os.environ.get("GROQ_API_KEY")

if not pinecone_key or not groq_key:
    raise ValueError("Missing API keys in .env file")

# 1. Initialize Pinecone
pc = Pinecone(api_key=pinecone_key)
index_name = "groq-hybrid-rag"

if not pc.has_index(index_name):
    print(f"Creating Pinecone index '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=384,  # Matches 'all-MiniLM-L6-v2' output dimension
        metric="dotproduct", # Required for hybrid search
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
index = pc.Index(index_name)

# 2. Initialize FREE Local Embedding Models
print("Loading local embedding models...")
# Dense (Semantic) Model via Hugging Face
dense_model = SentenceTransformer('all-MiniLM-L6-v2') 
# Sparse (Keyword) Model
bm25 = BM25Encoder()

# 3. Initialize Groq Client
groq_client = Groq(api_key=groq_key)

print("Environment setup complete.")

Loading local embedding models...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7358.30it/s]


Environment setup complete.


In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [22]:
import pdfplumber

pdf_path = r"D:\langchain bots\LANGCHAIN_BOTS-1\hybrid pincode search\NIPS-2017-attention-is-all-you-need-Paper.pdf" # REPLACE with your actual PDF
full_text = ""

# Extract text
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if text:
            full_text += text + "\n"

# Chunk the text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_text(full_text)
print(f"Extracted {len(full_text)} characters and split into {len(chunks)} chunks.")

Extracted 29255 characters and split into 65 chunks.


In [23]:
# 1. Fit the BM25 model on our specific PDF chunks
bm25.fit(chunks)
print("BM25 fitted to the document.")

# 2. Prepare data for Pinecone
records = []
for i, chunk in enumerate(chunks):
    # Generate vectors locally (FREE)
    dense_vec = dense_model.encode(chunk).tolist()
    sparse_vec = bm25.encode_documents(chunk)
    
    # Store the actual text in metadata so the LLM can read it later
    records.append({
        "id": f"chunk_{i}",
        "values": dense_vec,
        "sparse_values": sparse_vec,
        "metadata": {"text": chunk}
    })

# 3. Upsert to Pinecone
batch_size = 50
print("Upserting vectors to Pinecone...")
for i in range(0, len(records), batch_size):
    index.upsert(vectors=records[i:i + batch_size])
    
print("Ingestion complete!")

100%|██████████| 65/65 [00:00<00:00, 1205.27it/s]


BM25 fitted to the document.
Upserting vectors to Pinecone...
Ingestion complete!


In [24]:
# 1. The User's Question
user_question = "What is the main conclusion of the document?"

# 2. Generate Query Vectors (Locally)
dense_query = dense_model.encode(user_question).tolist()
sparse_query = bm25.encode_queries(user_question)

# 3. Apply Alpha Weighting (0.5 = 50% keyword, 50% semantic)
alpha = 0.5
weighted_dense = [v * alpha for v in dense_query]
weighted_sparse = {
    "indices": sparse_query["indices"],
    "values": [v * (1 - alpha) for v in sparse_query["values"]]
}

# 4. Search Pinecone
print("Searching database...")
search_results = index.query(
    vector=weighted_dense,
    sparse_vector=weighted_sparse,
    top_k=3, # Get the top 3 most relevant chunks
    include_metadata=True
)

# 5. Extract the retrieved text to build the context
retrieved_context = "\n\n".join([match['metadata']['text'] for match in search_results['matches']])

# 6. Build the prompt for Groq
system_prompt = f"""You are a helpful assistant. Use ONLY the following retrieved context to answer the user's question. 
If the answer is not in the context, say "I cannot find the answer in the provided document."

Retrieved Context:
{retrieved_context}
"""

# 7. Generate Answer with Groq
print("Generating answer with Groq (Qwen-2.5-32B)...\n")
completion = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile", # Or "llama-3.3-70b-versatile"
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_question}
    ],
    temperature=0.1, # Keep temperature low for factual RAG tasks
    max_completion_tokens=500
)

# 8. Output the final result
print("-" * 50)
print(completion.choices[0].message.content)
print("-" * 50)

Searching database...
Generating answer with Groq (Qwen-2.5-32B)...



AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}